<h1 style="color:#56B4E9;">Week 4 - Day 4</h1>

<h2 style="color:#0072B2;">Feature Engineering & Hyperparameter Tuning</h2>

<h3 style="color:#009E73;">Breast Cancer Classification</h3>

This notebook continues the Breast Cancer classification workflow using the same KNN model selected earlier.

The notebook is fully self-contained, but the focus is specifically on today's topics:

- Understanding how feature engineering changes the representation of the problem.
- Creating meaningful engineered features based on relationships in the data.
- Measuring whether those features actually help the model.
- Distinguishing learned parameters from hyperparameters.
- Tuning hyperparameters systematically using GridSearchCV.
- Comparing the tuned model against an untuned baseline using cross-validation.

<span style="color:#009E73;"><b>Main goal:</b></span>  
Improve the model through better representation and systematic tuning, while understanding **why** each change helps or hurts.

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

In [2]:
data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

# Keep the same target definition used previously:
# 1 = Malignant
# 0 = Benign
y = pd.Series(
    (data.target == 0).astype(int),
    name="Malignant"
)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)

Dataset shape: (569, 30)
Target shape: (569,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Final test shape:", X_test.shape)

Training shape: (455, 30)
Final test shape: (114, 30)


<h2 style="color:#0072B2;">Why Keep the Final Test Set Untouched?</h2>

Feature engineering and hyperparameter tuning both involve making decisions about the model.

If I repeatedly evaluate these decisions using the final test set, information from that test set begins to influence the development process.

For this reason:

- Feature-engineering decisions will be evaluated using the training data and cross-validation.
- Hyperparameters will also be selected using cross-validation.
- The final test set remains untouched during development.

<span style="color:#009E73;"><b>Connection to previous days:</b></span>  
Cross-validation is not only an evaluation technique. In today's workflow, it becomes the mechanism used to compare feature and hyperparameter choices without tuning against the final test set.

<h2 style="color:#0072B2;">What Is Feature Engineering?</h2>

Feature engineering is the process of creating or transforming input variables so that useful relationships in the data become easier for the model to learn.

A new feature can generally be represented as:

$$
z = f(x_1, x_2, \dots, x_k)
$$

where existing variables are transformed or combined to create a potentially more useful representation.

The goal is therefore not simply to increase the number of columns.

Instead, I want to improve the **representation of the problem**.

A useful way to think about the process is:

**Domain reasoning → Feature hypothesis → Experiment → Evidence**

<span style="color:#009E73;"><b>Key idea:</b></span>  
A new feature should have a reason to exist. I should be able to explain what information it represents before checking whether it improves model performance.

<h2 style="color:#0072B2;">Common Types of Feature Engineering</h2>

Feature engineering includes several different operations. They solve different representation problems and should not be applied blindly.

### 1. Feature Creation

Existing features can be combined to expose a relationship that is not explicitly represented as its own variable.

For example:

$$
z = \frac{x_1}{x_2}
$$

or

$$
z = x_1x_2
$$

or

$$
z = x_1 - x_2
$$

The usefulness of the engineered feature depends on whether the new relationship contains meaningful predictive structure.

---

### 2. Feature Transformation

An existing variable can be represented using a mathematical transformation.

For example:

$$
x' = \log(1+x)
$$

A transformation may make an underlying pattern easier for a model to use, especially when the original distribution has undesirable properties such as strong skewness.

---

### 3. Binning

A continuous variable can be divided into discrete groups.

For example:

**Age → Child / Adult / Senior**

Conceptually:

$$
AgeGroup(x)=
\begin{cases}
\text{Child} & x < 18 \\
\text{Adult} & 18 \leq x < 65 \\
\text{Senior} & x \geq 65
\end{cases}
$$

Binning can expose meaningful thresholds, but it also removes some numerical information.

For example, two values such as 25 and 60 may both become `Adult`, even though their original values were very different.

---

### 4. Encoding

Categorical variables often need to be transformed into a numerical representation.

Using arbitrary values such as:

**Nablus = 1, Hebron = 2, Ramallah = 3**

would create an artificial numerical relationship between the categories.

One-hot encoding instead creates separate indicator variables for the categories without implying an artificial ordering.

---

### 5. Datetime Feature Extraction

A datetime value may contain several hidden signals.

For example:

**2026-08-12 08:30**

can be decomposed into features such as:

- month
- day of week
- hour
- weekend indicator

This allows the model to use temporal patterns that were hidden inside the original datetime representation.

---

### 6. Scaling

Numeric features can have very different scales.

Standardization transforms a feature using:

$$
z = \frac{x-\mu}{\sigma}
$$

where:

- $x$ = original value
- $\mu$ = feature mean
- $\sigma$ = feature standard deviation
- $z$ = standardized value

Scaling is especially important for distance-based algorithms such as KNN because large numerical scales can otherwise dominate the distance calculation.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Different feature-engineering techniques solve different problems. I should use a technique because it matches the structure of the data and the model, not simply because the technique exists.

<h2 style="color:#0072B2;">Which Feature Engineering Techniques Are Relevant Here?</h2>

Not every feature-engineering technique is relevant to the Breast Cancer dataset.

For the current problem:

- There are no raw datetime variables, so datetime extraction is not needed.
- The input features are already numeric, so categorical encoding is not required.
- KNN is distance-based, so scaling remains important.
- Several measurements describe related geometric properties of cell nuclei, making feature creation especially interesting.

Therefore, the main experiment today will focus on creating relationships between existing measurements and testing whether those relationships improve the model.

<span style="color:#E69F00;"><b>Important:</b></span>  
I will not add transformations only to satisfy a checklist. Every engineered feature should represent a specific hypothesis about the data.

<h2 style="color:#0072B2;">Example Feature Hypothesis: Relative Radius</h2>

The dataset contains both:

- `mean radius`
- `worst radius`

In this dataset, `worst radius` is not a single before/after or time-based measurement. It is the **mean of the three largest radius measurements** for a sample.

Instead of considering only the absolute values, I can express how large this worst-radius summary is relative to the average radius:

$$
\text{radius ratio}
=
\frac{\text{worst radius}}
{\text{mean radius}}
$$

For example, if:

$$
\text{mean radius}=10
$$

and:

$$
\text{worst radius}=15
$$

then:

$$
\text{radius ratio}
=
\frac{15}{10}
=
1.5
$$

This means that the `worst radius` summary is **1.5 times the mean radius**, or 50% larger.

<span style="color:#009E73;"><b>Feature hypothesis:</b></span>  
The relative difference between the average radius and the largest-radius summary may expose a useful relationship that is not represented as explicitly when the two measurements are considered separately.


<h2 style="color:#0072B2;">Can an Engineered Feature Help or Hurt?</h2>

An engineered feature does not necessarily introduce new raw information.

For example:

$$
\text{radius ratio}
=
\frac{\text{worst radius}}
{\text{mean radius}}
$$

is calculated entirely from two existing features. Its purpose is to make their **relative relationship explicit**.

However, adding derived features is not automatically beneficial.

For KNN, every added feature becomes another dimension in the distance calculation:

$$
d(A,B)
=
\sqrt{
\sum_{j=1}^{p}(A_j-B_j)^2
}
$$

If many new features are derived from the same original measurement, that information may influence several dimensions and effectively receive more weight in the distance calculation.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Feature engineering should be treated as a hypothesis and tested experimentally rather than assumed to improve the model.

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=7))
])

baseline_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Baseline fold scores:", baseline_scores.round(4))
print(f"Mean F1: {baseline_scores.mean():.4f}")
print(f"Standard deviation: {baseline_scores.std():.4f}")

Baseline fold scores: [0.9538 0.9706 0.9538 0.9394 0.9697]
Mean F1: 0.9575
Standard deviation: 0.0116


In [5]:
X_train_engineered = X_train.copy()

X_train_engineered["radius ratio"] = (
    X_train_engineered["worst radius"]
    / X_train_engineered["mean radius"]
)

X_train_engineered[
    ["mean radius", "worst radius", "radius ratio"]
].head()

,mean radius,worst radius,radius ratio
10,16.02,19.19,1.197878
170,12.32,13.50,1.095779
407,12.85,14.40,1.120623
430,14.90,16.35,1.097315
27,18.61,21.31,1.145083


In [6]:
engineered_scores = cross_val_score(
    baseline_model,
    X_train_engineered,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Engineered fold scores:", engineered_scores.round(4))
print(f"Mean F1: {engineered_scores.mean():.4f}")
print(f"Standard deviation: {engineered_scores.std():.4f}")

print(
    f"\nChange in Mean F1: "
    f"{engineered_scores.mean() - baseline_scores.mean():+.4f}"
)

Engineered fold scores: [0.9538 0.9706 0.9538 0.9394 0.9851]
Mean F1: 0.9605
Standard deviation: 0.0157

Change in Mean F1: +0.0031


<h2 style="color:#0072B2;">Did the Radius Ratio Help?</h2>

Adding `radius ratio` changed the cross-validation result from:

- **Baseline:** 0.9575 ± 0.0116
- **With radius ratio:** 0.9605 ± 0.0157

The mean F1-score increased by **0.0031**, but the variation between folds also increased.

Most folds remained almost unchanged, while Fold 5 improved noticeably.

<span style="color:#009E73;"><b>Interpretation:</b></span>  
The engineered feature provided a small improvement in average performance, but the benefit was not consistent across all validation subsets.

Therefore, a higher mean alone is not enough to conclude that the new feature is universally better.

In [7]:
X_train_engineered["concavity gap"] = (
    X_train_engineered["worst concavity"]
    - X_train_engineered["mean concavity"]
)

X_train_engineered[
    ["mean concavity", "worst concavity", "concavity gap"]
].head()

,mean concavity,worst concavity,concavity gap
10,0.03299,0.1459,0.11291
170,0.03987,0.1242,0.08433
407,0.06126,0.1838,0.12254
430,0.27330,0.9019,0.62860
27,0.14900,0.3446,0.19560


<h2 style="color:#0072B2;">Second Engineered Feature: Concavity Gap</h2>

The second engineered feature is:

$$
\text{concavity gap}
=
\text{worst concavity}
-
\text{mean concavity}
$$

Here, `worst concavity` is the **mean of the three largest concavity measurements** for a sample.

A larger gap means that the largest-concavity summary is much higher than the average concavity.

<span style="color:#009E73;"><b>Feature hypothesis:</b></span>  
A larger difference between average concavity and the largest-concavity summary may capture additional information about irregularity in the nucleus shape.


In [8]:
two_feature_scores = cross_val_score(
    baseline_model,
    X_train_engineered,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Two-feature fold scores:", two_feature_scores.round(4))
print(f"Mean F1: {two_feature_scores.mean():.4f}")
print(f"Standard deviation: {two_feature_scores.std():.4f}")

print(
    f"\nChange vs Baseline: "
    f"{two_feature_scores.mean() - baseline_scores.mean():+.4f}"
)

print(
    f"Change vs Radius Ratio only: "
    f"{two_feature_scores.mean() - engineered_scores.mean():+.4f}"
)

Two-feature fold scores: [0.9538 0.9706 0.9375 0.9394 0.9552]
Mean F1: 0.9513
Standard deviation: 0.0121

Change vs Baseline: -0.0062
Change vs Radius Ratio only: -0.0092


In [9]:
X_train_concavity = X_train.copy()

X_train_concavity["concavity gap"] = (
    X_train_concavity["worst concavity"]
    - X_train_concavity["mean concavity"]
)

concavity_scores = cross_val_score(
    baseline_model,
    X_train_concavity,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Concavity-gap fold scores:", concavity_scores.round(4))
print(f"Mean F1: {concavity_scores.mean():.4f}")
print(f"Standard deviation: {concavity_scores.std():.4f}")

print(
    f"\nChange vs Baseline: "
    f"{concavity_scores.mean() - baseline_scores.mean():+.4f}"
)

Concavity-gap fold scores: [0.9538 0.9706 0.9538 0.9394 0.9552]
Mean F1: 0.9546
Standard deviation: 0.0099

Change vs Baseline: -0.0029


<h2 style="color:#0072B2;">Feature Engineering Results</h2>

The experiments produced:

| Feature Set | Mean F1 | Standard Deviation | Change vs Baseline |
|---|---:|---:|---:|
| Baseline | 0.9575 | 0.0116 | — |
| Radius ratio | 0.9605 | 0.0157 | +0.0031 |
| Concavity gap | 0.9546 | 0.0099 | -0.0029 |
| Radius ratio + Concavity gap | 0.9513 | 0.0121 | -0.0062 |

`radius ratio` achieved the highest mean F1-score.

Although the improvement was small, it provided the best result among the engineered features tested.

`concavity gap` produced a slightly lower mean F1-score but also a lower standard deviation, indicating lower fold-to-fold variability in this cross-validation run.

Combining both engineered features reduced performance further.

<span style="color:#009E73;"><b>Key lesson:</b></span>  
A feature that is logically meaningful does not necessarily improve predictive performance. Each engineered feature must be evaluated experimentally.

<span style="color:#009E73;"><b>Decision:</b></span>  
I will keep `radius ratio` as the engineered feature for the next modeling stage because it achieved the highest cross-validated mean F1-score.

In [10]:
X_train_final = X_train.copy()

X_train_final["radius ratio"] = (
    X_train_final["worst radius"]
    / X_train_final["mean radius"]
)

print("Final training shape:", X_train_final.shape)

Final training shape: (455, 31)


<h2 style="color:#0072B2;">Parameters vs. Hyperparameters</h2>

A <span style="color:#009E73;"><b>parameter</b></span> is learned by the model from the training data.

A <span style="color:#E69F00;"><b>hyperparameter</b></span> is chosen before training and controls how the model behaves or learns.

For a linear model:

$$
\hat{y} = w_0 + w_1x_1 + w_2x_2 + \dots + w_px_p
$$

The coefficients:

$$
w_0, w_1, w_2, \dots, w_p
$$

are **parameters** because they are estimated during `.fit()`.

For KNN, `n_neighbors = 7` in `KNeighborsClassifier(n_neighbors=7)` is a **hyperparameter** because it is chosen before training.

Other examples of hyperparameters include:

- `max_depth` in Decision Trees
- `n_estimators` in Random Forest
- `alpha` in Ridge and Lasso
- `C` in SVM

<span style="color:#009E73;"><b>Key idea:</b></span>  
Parameters are learned **during training**, while hyperparameters are selected **before training** and control the model's configuration.

<h2 style="color:#0072B2;">Why Add a Separate Untuned Baseline?</h2>

The earlier KNN configuration used `n_neighbors = 7`, which had already been selected in previous work.

For today's hyperparameter-tuning comparison, I need a genuinely **untuned reference**.

Therefore, I use the default `KNeighborsClassifier()` on the selected feature set before running GridSearchCV.

<span style="color:#009E73;"><b>Key idea:</b></span>  
The feature-engineering baseline and the hyperparameter-tuning baseline answer different questions: the first isolates the effect of the new feature, while the second measures whether systematic tuning improves a default model configuration.


In [11]:
untuned_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

untuned_scores = cross_val_score(
    untuned_model,
    X_train_final,
    y_train,
    cv=cv,
    scoring="f1"
)

print("Untuned fold scores:", untuned_scores.round(4))
print(f"Untuned Mean F1: {untuned_scores.mean():.4f}")
print(f"Untuned Standard Deviation: {untuned_scores.std():.4f}")

Untuned fold scores: [0.9538 0.9851 0.9375 0.9231 0.9697]
Untuned Mean F1: 0.9538
Untuned Standard Deviation: 0.0221


<h2 style="color:#0072B2;">What Does GridSearchCV Do?</h2>

`GridSearchCV` systematically searches through predefined hyperparameter values.

For every hyperparameter combination, it evaluates the model using cross-validation.

The process is:

**Define candidates → Try every combination → Cross-validate → Compare → Select the best**

If the grid contains several hyperparameters, the number of combinations is:

$$
N_{\text{combinations}}
=
N_1 \times N_2 \times \dots \times N_m
$$

With $k$-fold cross-validation:

$$
N_{\text{CV fits}}
=
N_{\text{combinations}} \times k
$$

For example, if KNN tests:

- `n_neighbors = [3, 5, 7, 9, 11]`
- `weights = ["uniform", "distance"]`

then there are:

$$
5 \times 2 = 10
$$

different hyperparameter combinations.

With 5-fold cross-validation:

$$
10 \times 5 = 50
$$

CV model fits are required to evaluate the grid.

`GridSearchCV` then refits the best configuration on the full training data by default; the **50** above refers specifically to the cross-validation fits.

<span style="color:#009E73;"><b>Key idea:</b></span>  
GridSearchCV replaces manual trial-and-error with a systematic comparison using the same cross-validation procedure for every candidate.

In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"]
}

grid_search = GridSearchCV(
    estimator=untuned_model,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train_final, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best CV F1: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

Best parameters: {'knn__n_neighbors': 7, 'knn__weights': 'uniform'}
Best CV F1: 0.9605


In [13]:
grid_results = pd.DataFrame(grid_search.cv_results_)

comparison = grid_results[
    [
        "param_knn__n_neighbors",
        "param_knn__weights",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

comparison

,param_knn__n_neighbors,param_knn__weights,mean_test_score,std_test_score,rank_test_score
5,7,distance,0.960550,0.015746,1
4,7,uniform,0.960550,0.015746,1
3,5,distance,0.953839,0.022093,3
2,5,uniform,0.953839,0.022093,3
0,3,uniform,0.953459,0.031347,5
1,3,distance,0.953459,0.031347,5
7,9,distance,0.947197,0.025777,7
9,11,distance,0.947197,0.025777,7
6,9,uniform,0.944121,0.021664,9
8,11,uniform,0.940639,0.026103,10


In [14]:
neighbor_effect = comparison.groupby(
    "param_knn__n_neighbors"
)["mean_test_score"].mean()

weight_effect = comparison.groupby(
    "param_knn__weights"
)["mean_test_score"].mean()

neighbor_range = neighbor_effect.max() - neighbor_effect.min()
weight_range = weight_effect.max() - weight_effect.min()

print("Mean F1 by n_neighbors:")
print(neighbor_effect.round(4))

print("\nMean F1 by weights:")
print(weight_effect.round(4))

print(f"\nPerformance range across n_neighbors: {neighbor_range:.4f}")
print(f"Performance range across weights: {weight_range:.4f}")

Mean F1 by n_neighbors:
param_knn__n_neighbors
3     0.9535
5     0.9538
7     0.9605
9     0.9457
11    0.9439
Name: mean_test_score, dtype: float64

Mean F1 by weights:
param_knn__weights
distance    0.9524
uniform     0.9505
Name: mean_test_score, dtype: float64

Performance range across n_neighbors: 0.0166
Performance range across weights: 0.0019


<h2 style="color:#0072B2;">Interpreting the GridSearch Results</h2>

The best cross-validated F1-score was:

**0.9605**

with:

- `n_neighbors = 7`
- `weights = uniform`

However, `weights="distance"` achieved exactly the same score when `n_neighbors=7`.

The quantitative comparison shows that <span style="color:#009E73;"><b>`n_neighbors` had the stronger effect</b></span> on performance:

- The mean-score range across `n_neighbors` values was substantially larger.
- The mean-score range across `weights` values was much smaller.

<span style="color:#E69F00;"><b>Important:</b></span>  
A value appearing in `best_params_` does not necessarily mean every selected hyperparameter was strongly better. Here, both weighting strategies tied at the best value of `k`.

<span style="color:#009E73;"><b>Key lesson:</b></span>  
`n_neighbors` was the most influential hyperparameter in this search, and `k = 7` provided the strongest performance among the tested values.


<h2 style="color:#0072B2;">Untuned Baseline vs. Tuned Model</h2>

Using the selected `radius ratio` feature, the default untuned KNN model achieved:

**Mean F1 = 0.9538 ± 0.0221**

GridSearchCV selected:

- `n_neighbors = 7`
- `weights = uniform`

with:

**Mean F1 = 0.9605 ± 0.0157**

The improvement in mean F1 was:

$$
0.9605 - 0.9538 = 0.0067
$$

The tuned configuration also showed lower fold-to-fold variability than the untuned reference in this cross-validation run.

<span style="color:#009E73;"><b>Decision:</b></span>  
GridSearchCV confirmed `n_neighbors = 7` as the strongest tested value and improved both the average F1-score and fold-to-fold consistency relative to the default untuned KNN configuration.


<h2 style="color:#0072B2;">GridSearchCV vs. RandomizedSearchCV</h2>

`GridSearchCV` evaluates **every possible combination** in the hyperparameter grid.

This works well when the search space is relatively small, but the computational cost grows quickly as the number of candidate values increases.

For example, suppose four hyperparameters each have 10 possible values:

$$
N_{\text{combinations}}
=
10 \times 10 \times 10 \times 10
=
10{,}000
$$

With 5-fold cross-validation:

$$
N_{\text{CV fits}}
=
10{,}000 \times 5
=
50{,}000
$$

`RandomizedSearchCV` does not evaluate every combination. Instead, it samples a specified number of combinations.

The number of sampled combinations is controlled by `n_iter`.

For example:

`RandomizedSearchCV(..., n_iter=50, cv=5)`

tests only **50 hyperparameter combinations**.

Therefore:

$$
N_{\text{CV fits}}
=
50 \times 5
=
250
$$

instead of 50,000 CV fits.

<span style="color:#009E73;"><b>Key idea:</b></span>  
Use `GridSearchCV` when the search space is small enough to test exhaustively. Use `RandomizedSearchCV` when the search space is large and evaluating every combination would be computationally expensive.

<span style="color:#E69F00;"><b>Important:</b></span>  
`n_iter` controls how many hyperparameter combinations RandomizedSearchCV samples, while `cv` controls how many cross-validation folds are used to evaluate each sampled combination.

In [15]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=untuned_model,
    param_distributions=param_grid,
    n_iter=5,
    cv=cv,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

<h2 style="color:#0072B2;">Final Takeaway</h2>

Feature engineering and hyperparameter tuning solve two different parts of the modeling problem.

Feature engineering changes **what information the model receives**, while hyperparameter tuning changes **how the model is configured to use that information**.

In this experiment:

- `radius ratio` was the most useful engineered feature.
- It increased the feature-engineering baseline mean F1 from **0.9575 to 0.9605**.
- `concavity gap` did not improve the mean F1-score.
- A separate default KNN reference achieved **0.9538 ± 0.0221** on the selected feature set.
- GridSearchCV selected `n_neighbors = 7` and achieved **0.9605 ± 0.0157**.
- Quantitative comparison showed that `n_neighbors` affected performance more strongly than `weights`.
- RandomizedSearchCV is more appropriate when the hyperparameter search space becomes too large for exhaustive grid search.

<span style="color:#009E73;"><b>Key lesson:</b></span>  
Better modeling is not about adding more features or testing more settings blindly. Each change should represent a clear hypothesis and be evaluated using reliable cross-validation evidence.

<span style="color:#D55E00;"><b>Evaluation discipline:</b></span>  
The final test set remained untouched throughout feature engineering and hyperparameter tuning.
